# G1 bám vận tốc 1,5 m/s trong 2000 vòng: PPO, Replay-IS, MPC-DAgger, Replay-IS + DAgger, MPC-Inject (Colab, GPU)

Train 5 phương pháp trên task **`Mjlab-Velocity-Flat-Unitree-G1-2k`** bằng lệnh `train` chuẩn
của mjlab (4096 env, **2000 vòng**), rồi đo vận tốc thực tế của từng policy khi ra lệnh đi
thẳng 0,5 / 1,0 / **1,5 m/s**.

**Task G1-2k khác task gốc ở đâu:** task gốc train 30.000 vòng và chỉ ra lệnh trên 1 m/s từ vòng
5000 (tối đa 3 m/s). G1-2k dừng ở 2000 vòng và mở rộng lệnh sớm hơn:

| Vòng | Lệnh vx (m/s) | Lệnh quay (rad/s) |
|---|---|---|
| 0 – 500 | −1,0 → 1,0 | ±0,5 |
| 500 – 2000 | −1,0 → **1,5** | ±0,7 |

| Phương pháp | Cách dùng MPC |
|---|---|
| **PPO** | Không dùng MPC (mốc so sánh = code gốc) |
| **Replay-IS** | Không dùng MPC; dùng lại 4 rollout cũ của chính PPO, hiệu chỉnh IS + V-trace |
| **Replay-IS+DAgger** | Kết hợp hai cách: dữ liệu PPO cũ vào loss PPO (hiệu chỉnh IS), nhãn MPC chỉ vào loss bắt chước |
| **MPC-DAgger** | Các env phụ do policy tự chạy; MPOPI gắn nhãn trạng thái policy đi qua; loss bắt chước giảm từ 1 về 0 |
| **MPC-Inject** | MPOPI tự chạy các env phụ (không nhiễu); mỗi batch PPO có 25% là mẫu MPC, coi như dữ liệu của chính PPO |

Mọi phương pháp dùng cùng **mức sàn std của policy** (`MIN_ACTION_STD`) để tránh lỗi NaN đã gặp với
MPC-Inject; mức sàn nhỏ nên hầu như không ảnh hưởng PPO.

**Lưu ý quan trọng:** chưa kiểm chứng được MPC có tự đi được G1 hay không (notebook
`mpc_g1_check_colab.ipynb`). Nếu MPC chỉ đứng yên, hai phương pháp MPC sẽ học theo một "thầy"
đứng yên ở đầu quá trình train.

**Thời gian:** khoảng 1,5 giờ cho mỗi lần train PPO trên GPU loại L4; các phương pháp có MPC chậm
hơn trong ~110 vòng đầu vì phải chạy MPC, các phương pháp có Replay-IS chậm hơn một chút vì batch
lớn gấp đôi. **5 phương pháp × 1 seed ≈ 8 giờ**, có thể vượt giới hạn một phiên Colab: bỏ chọn
bớt phương pháp ở mục 0 hoặc chạy làm nhiều phiên (các lần đã xong được bỏ qua). Nên dùng **L4 hoặc A100**. Mỗi lần train xong được
lưu lại trên Drive; nếu Colab ngắt, chạy lại mục 6 thì các lần đã xong được bỏ qua (lần đang
dở sẽ train lại từ đầu).

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpopi-before-module"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
TASK = "Mjlab-Velocity-Flat-Unitree-G1-2k"
NUM_ENVS = 4096  #@param {type:"integer"}
ITERATIONS = 2000  #@param {type:"integer"}
#@markdown Seed, cách nhau bằng dấu cách (mỗi seed × 5 phương pháp ≈ 8 giờ trên L4):
SEEDS = "1"  #@param {type:"string"}
#@markdown Phương pháp cần train:
TRAIN_PPO = True  #@param {type:"boolean"}
TRAIN_REPLAY_IS = True  #@param {type:"boolean"}
TRAIN_DAGGER = True  #@param {type:"boolean"}
TRAIN_REPLAY_IS_DAGGER = True  #@param {type:"boolean"}
TRAIN_INJECT = True  #@param {type:"boolean"}
#@markdown Mức sàn std của policy, dùng chung cho mọi phương pháp:
MIN_ACTION_STD = 0.05  #@param {type:"number"}
#@markdown **MPC (thầy):** số env phụ × số bước mỗi lần thu, thu mỗi `MPC_EVERY` vòng trong
#@markdown `MPC_ITERATIONS` vòng đầu. Planner MPOPI: `K` mẫu × `L` vòng, tầm nhìn `H` bước.
MPC_ENVS = 64  #@param {type:"integer"}
MPC_STEPS = 24  #@param {type:"integer"}
MPC_EVERY = 5  #@param {type:"integer"}
MPC_ITERATIONS = 110  #@param {type:"integer"}
PLANNER_K = 16  #@param {type:"integer"}
PLANNER_L = 2  #@param {type:"integer"}
PLANNER_H = 16  #@param {type:"integer"}
INJECT_FRACTION = 0.25  #@param {type:"number"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpc_g1_train"
else:
  OUT_ROOT = "/content/mpc_g1_train"
LOG_ROOT = f"{OUT_ROOT}/train_logs"
CONSOLE_DIR = f"{OUT_ROOT}/train_console"
EVAL_DIR = f"{OUT_ROOT}/eval"
os.makedirs(CONSOLE_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
SEED_LIST = [int(s) for s in SEEDS.split()]
N_SEGMENTS = -(-MPC_ITERATIONS // MPC_EVERY)
print("Kết quả lưu tại:", OUT_ROOT, "| seed:", SEED_LIST)
print(f"PPO: {NUM_ENVS} env × {ITERATIONS} vòng = {NUM_ENVS * 24} mẫu mỗi vòng")
print(f"MPC: {N_SEGMENTS} lần thu × {MPC_ENVS} env × {MPC_STEPS} bước = "
      f"{N_SEGMENTS * MPC_ENVS * MPC_STEPS} mẫu MPC")

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import subprocess


def run(cmd: str, cwd: str | None = None, quiet_tables: bool = False) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    if quiet_tables and ("[INFO]" in line or line.startswith(("|", "+"))):
      continue
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

Nhánh `mpc-stage1` phải được push bản mới nhất (có task `G1-2k` và Replay-IS + DAgger).

In [ ]:
import shutil

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
init_src = open(f"{REPO_DIR}/src/mjlab/tasks/velocity/config/g1/__init__.py").read()
config_src = open(f"{REPO_DIR}/src/mjlab/rl/mpopi/config.py").read()
if TASK not in init_src or "replay_own_rollouts" not in config_src:
  raise RuntimeError(
    f"Code trên GitHub đã cũ (thiếu task {TASK} hoặc tùy chọn replay_own_rollouts). "
    "Hãy push bản mới nhất của nhánh mpc-stage1 rồi chạy lại ô này."
  )
print("OK")

## 3. Cài đặt môi trường

Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch; print('torch', torch.__version__, 'cuda', "
  "torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)

## 4. Lệnh train của từng phương pháp

Mọi phương pháp dùng cùng lệnh `uv run train`, chỉ khác các tùy chọn
`--agent.algorithm.mpopi...`. Ô này in ra lệnh đầy đủ để bạn có thể chạy tay.

In [ ]:
A = "--agent.algorithm.mpopi"
COMMON = f"{A}.min-action-std {MIN_ACTION_STD}"
MPC_TEACHER = (
  f"{A}.mode mpc_ppo {A}.mpc.num-envs {MPC_ENVS} {A}.mpc.num-steps {MPC_STEPS} "
  f"{A}.mpc.collect-every {MPC_EVERY} {A}.mpc.collect-iterations {MPC_ITERATIONS} "
  f"{A}.mpc.execution-std 0 {A}.mpc.planner.num-samples {PLANNER_K} "
  f"{A}.mpc.planner.iterations {PLANNER_L} {A}.mpc.planner.horizon {PLANNER_H} "
  f"{A}.mpc.planner.noise-std 0.2 {A}.mpc.planner.num-knots 4"
)
DAGGER = (
  f"{MPC_TEACHER} {A}.mpc.use-in-ppo False {A}.mpc.driver policy "
  f"{A}.mpc.bc-coef 1.0 {A}.mpc.bc-iterations {MPC_ITERATIONS + 40} "
  f"{A}.mpc.max-age 50 {A}.mpc.buffer-segments 10"
)
METHODS = {}
if TRAIN_PPO:
  METHODS["PPO"] = f"{A}.mode ppo"
if TRAIN_REPLAY_IS:
  METHODS["Replay-IS"] = f"{A}.mode mpopi_ppo"
if TRAIN_DAGGER:
  METHODS["MPC-DAgger"] = DAGGER
if TRAIN_REPLAY_IS_DAGGER:
  METHODS["Replay-IS+DAgger"] = f"{DAGGER} {A}.mpc.replay-own-rollouts True"
if TRAIN_INJECT:
  METHODS["MPC-Inject"] = (
    f"{MPC_TEACHER} {A}.mpc.correction False {A}.mpc.bc-coef 0 "
    f"{A}.mpc.inject-fraction {INJECT_FRACTION} {A}.mpc.max-age None "
    f"{A}.mpc.buffer-segments {N_SEGMENTS}"
  )


def train_cmd(method: str, seed: int) -> str:
  return (
    f"uv run train {TASK} --agent.logger tensorboard --env.scene.num-envs {NUM_ENVS} "
    f"--agent.max-iterations {ITERATIONS} --agent.seed {seed} "
    f"--agent.run-name {method}_s{seed} --log-root {LOG_ROOT} {COMMON} {METHODS[method]}"
  )


for m in METHODS:
  print(f"# {m}\n{train_cmd(m, SEED_LIST[0])}\n")

## 5. Chạy thử nhanh (3 vòng mỗi phương pháp)

Kiểm tra cả 3 phương pháp chạy được trên GPU và đo thời gian một lần thu MPC. Nếu ô này lỗi
thì dừng lại.

In [ ]:
import re
import time

for m in METHODS:
  t0 = time.time()
  cmd = train_cmd(m, 0).replace(f"--agent.max-iterations {ITERATIONS}", "--agent.max-iterations 3")
  cmd = cmd.replace(f"--log-root {LOG_ROOT}", f"--log-root {OUT_ROOT}/smoke_logs")
  cmd = cmd.replace(f"collect-every {MPC_EVERY} ", "collect-every 1 ")
  log = f"{OUT_ROOT}/smoke_{m}.log"
  run(f"{cmd} > {log} 2>&1 || (tail -30 {log}; exit 1)", cwd=REPO_DIR)
  text = open(log, errors="ignore").read()
  it_times = [float(x) for x in re.findall(r"Iteration time:\s*([\d.]+)s", text)]
  mpc = [float(x) for x in re.findall(r"mpc/collect_seconds loss:\s*([\d.]+)", text)]
  info = f"vòng cuối {it_times[-1]:.1f} s" if it_times else ""
  if mpc:
    info += f", một lần thu MPC {max(mpc):.0f} s"
  print(f"OK {m}: tổng {time.time() - t0:.0f} s ({info})")

## 6. Train (chạy nền)

Các lần train chạy **lần lượt** (4096 env đã dùng hết GPU). Mỗi lần xong để lại file `.done`;
chạy lại ô này sau khi Colab ngắt thì các lần đã xong được bỏ qua.

In [ ]:
import shlex
import signal

if "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS):
  raise RuntimeError("Đang có train chạy nền. Dừng ở mục 7.2 trước.")

jobs = [(m, s) for s in SEED_LIST for m in METHODS]
lane = []
for m, s in jobs:
  name = f"{m}_s{s}"
  log = shlex.quote(f"{CONSOLE_DIR}/{name}.log")
  done = shlex.quote(f"{CONSOLE_DIR}/{name}.done")
  lane.append(f"[ -f {done} ] || ({train_cmd(m, s)} > {log} 2>&1 && touch {done})")
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
TRAIN_PROCS = [
  subprocess.Popen(["bash", "-c", "; ".join(lane)], cwd=REPO_DIR, env=env, start_new_session=True)
]
print(f"Đã bắt đầu {len(jobs)} lần train (lần lượt):")
for m, s in jobs:
  print(f"  {m} seed {s}")

### 7.1 Xem tiến độ

Chạy lại ô này bất cứ lúc nào. In vòng mới nhất, thời gian còn lại và điểm bám vận tốc
thẳng (`track_linear_velocity`, tối đa 2,0) của từng lần train, và vẽ đường học. Đường nét đứt
ở vòng 500 là lúc lệnh bắt đầu lên tới 1,5 m/s.

In [ ]:
#@title Xem tiến độ
import glob

import matplotlib.pyplot as plt

ANSI = re.compile(r"\x1b\[[0-9;]*m")


def parse_log(path):
  """Per-iteration (it, mean reward, track_linear_velocity), ETA and the last error."""
  text = ANSI.sub("", open(path, errors="ignore").read())
  rows, last_it = [], None
  for block in text.split("Learning iteration")[1:]:
    it = re.match(r"\s*(\d+)/", block)
    if not it:
      continue
    last_it = int(it.group(1))
    rew = re.search(r"Mean reward:\s*(-?[\d.]+)", block)
    track = re.search(r"Episode_Reward/track_linear_velocity:\s*(-?[\d.]+)", block)
    if rew and track:
      rows.append((last_it, float(rew.group(1)), float(track.group(1))))
  eta = re.findall(r"ETA:\s*([\d:]+)", text)
  err = re.findall(r"^\w*Error: .*$", text, flags=re.M)
  return rows, last_it, eta[-1] if eta else None, err[-1] if err else None


COLORS = {
  "PPO": "#555555",
  "Replay-IS": "#2ca02c",
  "MPC-DAgger": "#d62728",
  "Replay-IS+DAgger": "#9467bd",
  "MPC-Inject": "#1f77b4",
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for path in sorted(glob.glob(f"{CONSOLE_DIR}/*.log")):
  name = os.path.basename(path).removesuffix(".log")
  method = name.rsplit("_s", 1)[0]
  rows, last_it, eta, err = parse_log(path)
  done = os.path.exists(path.replace(".log", ".done"))
  status = "XONG" if done else ("LỖI: " + err if err else f"đang chạy, còn {eta}")
  if last_it is None:
    last = "chưa xong vòng 0 (đang khởi tạo hoặc đang thu dữ liệu MPC)"
  elif rows:
    last = f"vòng {last_it}/{ITERATIONS}, reward {rows[-1][1]:.1f}, bám vận tốc {rows[-1][2]:.2f}/2"
  else:
    last = f"vòng {last_it}/{ITERATIONS}"
  age = time.time() - os.path.getmtime(path)
  print(f"{name:>20}: {last} | {status} | log cập nhật {age:.0f} s trước")
  if rows:
    its, rew, track = zip(*rows)
    axes[0].plot(its, rew, color=COLORS.get(method), alpha=0.8, label=name)
    axes[1].plot(its, track, color=COLORS.get(method), alpha=0.8, label=name)
running = "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS)
print("\nTrạng thái:", "ĐANG TRAIN" if running else "không có train nào đang chạy")
for ax, label in ((axes[0], "Mean reward"), (axes[1], "Bám vận tốc thẳng (tối đa 2,0)")):
  ax.axvline(500, color="k", ls="--", lw=0.8)
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(label)
  ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

### 7.2 Chờ train xong / dừng train

In [ ]:
#@title Chờ train xong (in tiến độ mỗi 5 phút; bấm Stop để ngừng chờ, train vẫn chạy)
while "TRAIN_PROCS" in globals() and any(p.poll() is None for p in TRAIN_PROCS):
  time.sleep(300)
  n_done = len(glob.glob(f"{CONSOLE_DIR}/*.done"))
  print(time.strftime("%H:%M"), f"{n_done}/{len(SEED_LIST) * len(METHODS)} lần train đã xong")
print("Train đã kết thúc.")

In [ ]:
#@title Dừng hẳn train
for p in TRAIN_PROCS if "TRAIN_PROCS" in globals() else []:
  if p.poll() is None:
    os.killpg(p.pid, signal.SIGTERM)
print("Đã gửi lệnh dừng.")

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$LOG_ROOT" --reload_interval 30

## 9. Đường học của 3 phương pháp

Đọc log TensorBoard, lấy trung bình theo seed: reward lúc train, điểm bám vận tốc thẳng, độ
dài episode (tối đa 1000 bước = 20 s).

In [ ]:
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

TAGS = {
  "Train/mean_reward": "Mean reward (lúc train)",
  "Episode_Reward/track_linear_velocity": "Bám vận tốc thẳng (tối đa 2,0)",
  "Train/mean_episode_length": "Độ dài episode (bước)",
}


def run_dir(method, seed):
  found = glob.glob(f"{LOG_ROOT}/**/*_{method}_s{seed}", recursive=True)
  return max(found, key=os.path.getmtime) if found else None


def load_scalars(path):
  ea = EventAccumulator(path, size_guidance={"scalars": 0})
  ea.Reload()
  have = ea.Tags()["scalars"]
  return {t: {e.step: e.value for e in ea.Scalars(t)} if t in have else {} for t in TAGS}


data = {}
for m in METHODS:
  for s in SEED_LIST:
    d = run_dir(m, s)
    if d:
      data[(m, s)] = load_scalars(d)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for m in METHODS:
  runs = [data[(m, s)] for s in SEED_LIST if (m, s) in data]
  for ax, tag in zip(axes, TAGS):
    curves = [r[tag] for r in runs if r[tag]]
    for c in curves:
      ks = sorted(c)
      ax.plot(ks, [c[k] for k in ks], color=COLORS[m], alpha=0.25, lw=1)
    if curves:
      common = sorted(set.intersection(*(set(c) for c in curves)))
      ax.plot(common, [np.mean([c[k] for c in curves]) for k in common], color=COLORS[m],
              lw=2.2, label=f"{m} ({len(curves)} seed)")
for ax, tag in zip(axes, TAGS):
  ax.axvline(500, color="k", ls="--", lw=0.8)
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(TAGS[tag])
  ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/train_curves.png", dpi=150)
plt.show()

## 10. Đo vận tốc thực tế ở 0,5 / 1,0 / 1,5 m/s

Lấy checkpoint cuối của mỗi lần train, cho policy (tất định, không nhiễu) điều khiển 16 robot
đi thẳng trong 5 giây ở từng lệnh vận tốc; bỏ 1 giây đầu (tăng tốc). Có kèm mốc "đứng yên".

**Tiêu chí "bám được 1,5 m/s"** (giống tiêu chí đã đặt cho MPC): vận tốc trung bình ≥ 1,2 m/s ở
lệnh 1,5 m/s và ≤ 0,25 lần ngã mỗi robot.

In [ ]:
#@title Đánh giá
EVAL_ENVS = 16  #@param {type:"integer"}
RECORD_VIDEO = True  #@param {type:"boolean"}


def last_checkpoint(d):
  ckpts = glob.glob(f"{d}/model_*.pt") if d else []
  return max(ckpts, key=lambda p: int(re.search(r"model_(\d+)", p).group(1))) if ckpts else None


for m in METHODS:
  for s in SEED_LIST:
    out = f"{EVAL_DIR}/{m}_s{s}.json"
    ckpt = last_checkpoint(run_dir(m, s))
    if os.path.exists(out) or ckpt is None:
      print(f"{m}_s{s}: " + ("đã có kết quả" if ckpt else "chưa có checkpoint") + ", bỏ qua")
      continue
    controllers = "zero policy" if (m, s) == (list(METHODS)[0], SEED_LIST[0]) else "policy"
    video = f"--video-dir {EVAL_DIR}/video/{m}_s{s}" if RECORD_VIDEO and s == SEED_LIST[0] else ""
    print(f"===== {m}_s{s}: {os.path.basename(ckpt)}")
    run(
      f"uv run python scripts/mpc/eval_mpc_velocity.py --task {TASK} --device cuda:0 "
      f"--num-envs {EVAL_ENVS} --steps 250 --speeds 0.5 1.0 1.5 --controllers {controllers} "
      f"--checkpoint {ckpt} --out {out} {video}",
      cwd=REPO_DIR, quiet_tables=True,
    )

In [ ]:
#@title Bảng kết quả và kết luận
import json

rows = []
for path in sorted(glob.glob(f"{EVAL_DIR}/*.json")):
  name = os.path.basename(path).removesuffix(".json")
  method = name.rsplit("_s", 1)[0]
  for r in json.load(open(path))["results"]:
    rows.append({"method": method if r["controller"] == "policy" else "đứng yên", "run": name, **r})

print(f"{'phương pháp':>16} {'lệnh':>5} {'vận tốc':>14} {'|sai số|':>9} {'ngã/robot':>10} {'seed':>5}")
summary = {}
for method in ["đứng yên", *METHODS]:
  for cmd in (0.5, 1.0, 1.5):
    rs = [r for r in rows if r["method"] == method and r["command"] == cmd]
    if not rs:
      continue
    v = np.array([r["speed"] for r in rs])
    spread = f" ± {v.std():.2f}" if len(v) > 1 else ""
    print(f"{method:>16} {cmd:>5.1f} {v.mean():>8.2f}{spread:<6} "
          f"{np.mean([r['abs_error'] for r in rs]):>9.2f} "
          f"{np.mean([r['falls_per_env'] for r in rs]):>10.2f} {len(rs):>5}")
    if cmd == 1.5 and method != "đứng yên":
      summary[method] = [(r["run"], r["speed"], r["falls_per_env"]) for r in rs]

print("\nTiêu chí bám 1,5 m/s (≥ 1,2 m/s và ≤ 0,25 lần ngã/robot):")
for method, runs in summary.items():
  ok = [run for run, v, f in runs if v >= 1.2 and f <= 0.25]
  print(f"  {method}: {len(ok)}/{len(runs)} lần train ĐẠT")

## 11. Video ở lệnh 1,5 m/s (robot đầu tiên, seed đầu tiên)

In [ ]:
from IPython.display import HTML, Video, display

for path in sorted(glob.glob(f"{EVAL_DIR}/video/*/policy_1.5mps.mp4")):
  display(HTML(f"<b>{os.path.relpath(path, EVAL_DIR)}</b>"))
  display(Video(path, embed=True, width=480))

## 12. Tải kết quả về

Chỉ gồm log console, TensorBoard, kết quả đánh giá, video và hình (không kèm checkpoint).

In [ ]:
PACK = "/content/mpc_g1_train_results"
if os.path.exists(PACK):
  shutil.rmtree(PACK)
shutil.copytree(
  OUT_ROOT, PACK, ignore=shutil.ignore_patterns("*.pt", "*.onnx", "smoke_logs")
)
shutil.make_archive(PACK, "zip", PACK)
from google.colab import files

files.download(f"{PACK}.zip")